# Busan Traffic AI PBL - 14_busan_air_quality_preprocessing.ipynb

부산시 교통량 예측 실습을 부산시 교통량 예측 프로젝트로 변경한 노트북입니다.
API 인증키와 ngrok token은 코드에 직접 넣지 말고 환경변수로 관리합니다.


In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/busan_air_2022_2023_sample.csv", encoding="cp949")
df.head()


In [ ]:
df.info()


In [ ]:
df = df.rename(columns={
    "측정일시": "date",
    "측정소명": "station",
    "이산화질소농도(ppm)": "NO2",
    "오존농도(ppm)": "O3",
    "일산화탄소농도(ppm)": "CO",
    "아황산가스농도(ppm)": "SO2",
    "미세먼지농도(㎍/㎥)": "PM10",
    "초미세먼지농도(㎍/㎥)": "PM25"
})

df.head()


In [ ]:
df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")
df.head()


In [ ]:
df_junggu = df[df["station"] == "중구"].copy()
df_junggu.head()


In [ ]:
df_junggu = df_junggu.drop(columns=["station"])
df_junggu.head()


In [ ]:
df_junggu.isnull().sum()


In [ ]:
df_junggu.describe()


In [ ]:
df_junggu.head()

In [ ]:
df_test = pd.read_csv("air_2023.csv", encoding="cp949")
df_test.head()

In [ ]:
df_test = df_test.rename(columns={
    "측정일시": "date",
    "측정소명": "station",
    "이산화질소농도(ppm)": "NO2",
    "오존농도(ppm)": "O3",
    "일산화탄소농도(ppm)": "CO",
    "아황산가스농도(ppm)": "SO2",
    "미세먼지농도(㎍/㎥)": "PM10",
    "초미세먼지농도(㎍/㎥)": "PM25"
})
df_test["date"] = pd.to_datetime(df_test["date"], format="%Y%m%d")
df_junggu_test = df_test[df_test["station"] == "중구"].copy()
df_junggu_test = df_junggu_test.drop(columns=["station"])
df_junggu_test.head()

In [ ]:
df_junggu = pd.concat([df_junggu, df_junggu_test], ignore_index=True)
df_junggu.head()

In [ ]:
import io

path = "ta_20260107204237.csv"
# 1) 파일 전체를 읽는다
with open(path, "r", encoding="cp949") as f:
    lines = f.readlines()

# 2) '날짜'와 '지점'이 포함된 줄을 찾아서 그 줄부터 데이터로 본다
header_idx = None
for i, line in enumerate(lines):
    if ("날짜" in line) and ("지점" in line):
        header_idx = i
        break

print("header_idx:", header_idx)
print("header line:", lines[header_idx])

In [ ]:
text = "".join(lines[header_idx:])

df_ta = pd.read_csv(io.StringIO(text), sep=",")
df_ta["날짜"] = df_ta["날짜"].astype(str).str.strip()
df_ta.head()
df_ta = df_ta.drop(columns=["지점"])
df_ta.head()

In [ ]:
import io

path = "rn_20260107204346.csv"
# 1) 파일 전체를 읽는다
with open(path, "r", encoding="cp949") as f:
    lines = f.readlines()

# 2) '날짜'와 '지점'이 포함된 줄을 찾아서 그 줄부터 데이터로 본다
header_idx = None
for i, line in enumerate(lines):
    if ("날짜" in line) and ("지점" in line):
        header_idx = i
        break

print("header_idx:", header_idx)
print("header line:", lines[header_idx])

In [ ]:
text = "".join(lines[header_idx:])

df_rn = pd.read_csv(io.StringIO(text), sep=",")
df_rn["날짜"] = df_rn["날짜"].astype(str).str.strip()
df_rn.head()
df_rn = df_rn.drop(columns=["지점"])
df_rn.head()

In [ ]:
df_rn.isnull().sum()

In [ ]:
df_rn.fillna(0, inplace=True)

In [ ]:
df_rn.head()

In [ ]:
import pandas as pd

holiday = pd.read_csv("holiday.csv", encoding="cp949")
holiday.head()

In [ ]:
holiday["휴일시작일"] = pd.to_datetime(holiday["휴일시작일"])
holiday["휴일종료일"] = pd.to_datetime(holiday["휴일종료일"])

holiday.info()

In [ ]:
holiday_dates = []

for _, row in holiday.iterrows():
    dates = pd.date_range(start=row["휴일시작일"], end=row["휴일종료일"])
    holiday_dates.extend(dates)

In [ ]:
holiday_df = pd.DataFrame({
    "date": holiday_dates,
    "is_holiday": 1
})

holiday_df.head()

In [ ]:
all_dates = pd.DataFrame({
    "date": pd.date_range(start="2022-01-01", end="2023-12-31")
})

holiday_full = all_dates.merge(
    holiday_df,
    on="date",
    how="left"
)

holiday_full["is_holiday"] = holiday_full["is_holiday"].fillna(0).astype(int)
holiday_full.head()

In [ ]:
import glob
import re

files = sorted(glob.glob("*_2022.xlsx") + glob.glob("*_2023.xlsx"))
files[:5], len(files)

In [ ]:
def load_one_file_daily_total(filepath):
    # 두 번째 시트 index=1
    df = pd.read_excel(filepath, sheet_name=1)

    # 컬럼명 공백 정리
    df.columns = df.columns.astype(str).str.strip()

    # 시간 컬럼 찾기: "0시" ~ "23시"
    hour_cols = [c for c in df.columns if re.fullmatch(r"\d{1,2}시", c)]
    hour_cols = sorted(hour_cols, key=lambda x: int(x.replace("시","")))

    # 숫자 변환 (빈칸, 문자열 등이 있을 수 있음)
    df[hour_cols] = df[hour_cols].apply(pd.to_numeric, errors="coerce").fillna(0)

    # 행 단위 24시간 합 (지점/방향/구분 무시)
    df["row_daily_total"] = df[hour_cols].sum(axis=1)

    # 일자 정리 (20220101 형태일 가능성이 큼)
    df["일자"] = df["일자"].astype(str).str.strip()

    # 일자별 합산 (모든 행 합)
    out = df.groupby("일자", as_index=False)["row_daily_total"].sum()
    out = out.rename(columns={"일자": "date", "row_daily_total": "traffic_total"})

    # date를 datetime으로 변환 (안 되면 문자열 유지)
    out["date"] = pd.to_datetime(out["date"], errors="coerce", format="%Y%m%d").fillna(pd.to_datetime(out["date"], errors="coerce"))
    return out

In [ ]:
all_daily = []

for f in files:
    try:
        tmp = load_one_file_daily_total(f)
        all_daily.append(tmp)
        print("ok", f, tmp.shape)
    except Exception as e:
        print("fail", f, e)

traffic_daily = pd.concat(all_daily, ignore_index=True)

# 같은 날짜가 여러 파일에서 중복되면 다시 한번 합산
traffic_daily = traffic_daily.groupby("date", as_index=False)["traffic_total"].sum()

traffic_daily = traffic_daily.sort_values("date").reset_index(drop=True)
traffic_daily.head(), traffic_daily.tail(), traffic_daily.shape


In [ ]:
traffic_daily.to_csv("data/processed/traffic_daily_total_2022_2023.csv", index=False, encoding="utf-8-sig")

In [ ]:
holiday_full["date"]

In [ ]:
df_ta = df_ta.rename(columns={"날짜": "date"})
df_rn = df_rn.rename(columns={"날짜": "date"})

In [ ]:
for df in [df_junggu, df_ta, df_rn, holiday_full]:
    df["date"] = pd.to_datetime(df["date"], errors="coerce")

In [ ]:
df_all = (
    df_junggu
    .merge(df_ta, on="date", how="left")
    .merge(df_rn, on="date", how="left")
    .merge(holiday_full, on="date", how="left")
)

In [ ]:
df_all.info()
df_all.head()

In [ ]:
df_all.to_csv("data/processed/merged_x.csv", index=False, encoding="utf-8-sig")